# SVDKL

SVDKL combines structural-variant selection, an attention-based feature encoder and Gaussian process regression to predict tomato metabolite traits. Choose a model capacity, load its weights and input data, then run prediction. Training and evaluation are optional.


In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/wyqmath/SVDKL.git"
try:
    in_colab = importlib.util.find_spec("google.colab") is not None
except (ModuleNotFoundError, ValueError):
    in_colab = False
required = ["src/model.py", "src/train.py", "src/predict.py", "src/score.py", "src/prepare_data.py"]
candidates = [Path.cwd(), Path.cwd().parent]
REPO_DIR = next((path for path in candidates if (path / "src/model.py").is_file()),
                Path("/content/SVDKL") if in_colab else Path.cwd() / "SVDKL")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
missing = [name for name in required if not (REPO_DIR / name).is_file()]
if missing:
    raise RuntimeError("The repository is incomplete: " + ", ".join(missing))
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
packages = ("numpy", "scipy", "torch", "threadpoolctl")
missing_packages = [name for name in packages if importlib.util.find_spec(name) is None]
if missing_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing_packages], check=True)
import numpy as np
import torch
from src.train import get_benchmark
from src import model as model_module

if not hasattr(model_module, "load_parameter_checkpoint"):
    raise RuntimeError("This checkout predates parameter-only weight support. Update the repository source files, restart the runtime and rerun setup.")

plan = get_benchmark()
print("Repository:", REPO_DIR)
for package in packages:
    print(package, importlib.metadata.version(package))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Model and weights

Choose the matching `SVDKL-0.01M_weights.zip`, `SVDKL-4.35M_weights.zip` or `SVDKL-46.46M_weights.zip`. The current ZIP contains 36 parameter files, one per trait. Upload the ZIP, enter an existing file path, or supply its public file URL. Complete-checkpoint ZIPs with their embedded input snapshot are also supported.

Parameter files require the fixed reference input to reconstruct training preprocessing and the Gaussian process state. In the **Input data** section, use an existing `dataset_inputs.npz` or prepare it from [706.sv.vcf.gz](http://solomics.agis.org.cn/tomato/ftp/genotypes/706.sv.vcf.gz) and [pheno_exp_meta.txt](http://solomics.agis.org.cn/tomato/ftp/expression/pheno_exp_meta.txt). These sources accompany [Zhou et al. (2022)](https://doi.org/10.1038/s41586-022-04808-9) and the metabolite measurements from [Zhu et al. (2018)](https://doi.org/10.1016/j.cell.2017.12.019). Original-file downloads start only when that input option is selected.


In [ ]:
MODEL = "SVDKL-0.01M"  # @param ["SVDKL-0.01M", "SVDKL-4.35M", "SVDKL-46.46M"]
print("Selected capacity:", MODEL)


In [ ]:
import filecmp
import re
import shutil
import stat
import tempfile
import urllib.parse
import urllib.request
import zipfile
from pathlib import PurePosixPath


def unpack_zip(archive_path, root):
    with zipfile.ZipFile(archive_path) as archive, tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        staging, entries, seen = Path(temporary), [], set()
        if root == "weights":
            prefix = f"weights/rebuilt_fold0/{MODEL}"
            complete = {"weights/marker_ids.npy", "weights/_input_snapshot.npz", prefix + "/index.json"}
            for trait in plan["traits"]:
                stem = f"{prefix}/fold_0/{MODEL}_f0_{trait}"
                complete.update({stem + ".full.pt", stem + ".json"})
            parameters = {f"weights/{MODEL}/{trait}.pt" for trait in plan["traits"]}
            members = [member.filename for member in archive.infolist() if not member.is_dir()]
            if len(members) != len(set(members)) or set(members) not in (parameters, complete):
                raise ValueError(f"Choose the {MODEL} ZIP: 36 parameter files, or 75 files including complete checkpoints, records and shared inputs.")
        for member in archive.infolist():
            name = member.orig_filename
            parts = name.rstrip("/").split("/")
            mode = member.external_attr >> 16
            if ("\\" in name or "\x00" in name or PurePosixPath(name).is_absolute()
                    or parts[0] != root or any(p in {"", ".", ".."} or ":" in p or p.rstrip(". ") != p for p in parts)
                    or stat.S_ISLNK(mode) or stat.S_IFMT(mode) not in {0, stat.S_IFREG, stat.S_IFDIR}):
                raise ValueError(f"Unsafe ZIP entry: {name!r}")
            key = "/".join(parts).casefold()
            if key in seen or (len(parts) == 1 and not member.is_dir()):
                raise ValueError(f"Invalid or duplicate ZIP entry: {name!r}")
            if root == "weights" and len(parts) > 1:
                common = len(parts) == 2 and parts[1] in {"marker_ids.npy", "_input_snapshot.npz"}
                rebuilt = (parts[1] == "rebuilt_fold0" and
                           ((len(parts) == 2 and member.is_dir()) or
                            (len(parts) >= 3 and parts[2] == MODEL)))
                parameter = parts[1] == MODEL
                if not (common or rebuilt or parameter):
                    raise ValueError(f"Select the {MODEL} weights ZIP for the chosen capacity.")
            seen.add(key)
            if member.is_dir():
                continue
            source, target = staging.joinpath(*parts), REPO_DIR.joinpath(*parts)
            source.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as handle, source.open("xb") as output:
                shutil.copyfileobj(handle, output)
            if target.is_symlink() or any(parent.is_symlink() for parent in target.parents):
                raise ValueError(f"Refusing a symbolic-link destination: {name!r}")
            if target.exists() and (not target.is_file() or not filecmp.cmp(source, target, shallow=False)):
                raise FileExistsError(f"Conflicting existing file: {name!r}; no files were replaced.")
            entries.append((source, target))
        if not entries:
            raise ValueError("The ZIP contains no files.")
        for source, target in entries:
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                # Staging is on the same filesystem: publish without a second
                # full-size copy and never replace an existing target.
                os.link(source, target)
                source.unlink()


def download_to(url, target):
    parsed = urllib.parse.urlsplit(url.strip())
    if parsed.scheme not in {"http", "https"} or not parsed.netloc:
        raise ValueError("Enter a public file URL or use file upload.")
    if parsed.hostname in {"drive.google.com", "docs.google.com"}:
        match = re.search(r"/file/d/([A-Za-z0-9_-]+)", parsed.path)
        file_id = match.group(1) if match else urllib.parse.parse_qs(parsed.query).get("id", [""])[0]
        if not re.fullmatch(r"[A-Za-z0-9_-]+", file_id):
            raise ValueError("Use a Google Drive file link, not a folder link.")
        if importlib.util.find_spec("gdown") is None:
            subprocess.run([sys.executable, "-m", "pip", "install", "gdown"], check=True)
        import gdown
        if gdown.download(id=file_id, output=str(target), use_cookies=False) is None:
            raise RuntimeError("The Google Drive file could not be downloaded.")
    else:
        with urllib.request.urlopen(url.strip(), timeout=60) as response, target.open("xb") as output:
            shutil.copyfileobj(response, output, length=1024 * 1024)


def receive_zip(label):
    from google.colab import files

    with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        archive_path = Path(temporary) / "archive.zip"
        print(f"Choose {label}")
        previous = Path.cwd()
        try:
            os.chdir(temporary)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1:
            raise ValueError("Upload one weights ZIP file.")
        archive_path.write_bytes(next(iter(uploaded.values())))
        unpack_zip(archive_path, "weights")


In [ ]:
WEIGHTS_SOURCE = "Upload ZIP"  # @param ["Upload ZIP", "Existing ZIP path", "Public URL", "Existing weights", "Train from scratch"]
EXISTING_ZIP_PATH = ""  # @param {type:"string"}
WEIGHTS_URL = ""  # @param {type:"string"}

if WEIGHTS_SOURCE == "Existing ZIP path":
    if not EXISTING_ZIP_PATH.strip():
        raise ValueError(f"Enter the path to {MODEL}_weights.zip, for example on a mounted Drive, or select Upload ZIP.")
    archive_path = Path(EXISTING_ZIP_PATH).expanduser()
    if not archive_path.is_file():
        raise FileNotFoundError(archive_path)
    unpack_zip(archive_path, "weights")
    print("Loaded:", archive_path)
elif WEIGHTS_SOURCE == "Upload ZIP":
    receive_zip(f"{MODEL}_weights.zip")
elif WEIGHTS_SOURCE == "Public URL":
    if not WEIGHTS_URL.strip():
        raise ValueError(f"Enter the public file link for {MODEL}_weights.zip or choose Upload ZIP.")
    with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        archive_path = Path(temporary) / "weights.zip"
        download_to(WEIGHTS_URL, archive_path)
        unpack_zip(archive_path, "weights")
    print("Loaded weights from the supplied URL.")
elif WEIGHTS_SOURCE == "Existing weights":
    parameter_files = [Path("weights") / MODEL / f"{trait}.pt" for trait in plan["traits"]]
    complete_files = [Path("weights/rebuilt_fold0") / MODEL / "fold_0" / f"{MODEL}_f0_{trait}.full.pt" for trait in plan["traits"]]
    if not (all(path.is_file() for path in parameter_files) or all(path.is_file() for path in complete_files)):
        raise FileNotFoundError(f"The 36 weights for {MODEL} are missing. Upload {MODEL}_weights.zip first.")
    print("Using existing weights for", MODEL)
else:
    print("Prepare the original input data below, then enable training before prediction.")


## Input data

For a parameter-only ZIP, upload an existing `dataset_inputs.npz` or select an original-file option below. Place the original files in `data/` through the Colab Files sidebar, upload them here, or explicitly start their download. Large VCF files are best uploaded through the sidebar or copied from mounted storage.

A complete-checkpoint ZIP can use its embedded `weights/_input_snapshot.npz` directly. Prepared inputs contain 324 accessions, 106,362 DELs and 36 traits; their IDs and order are checked before use. When an independent snapshot is available, all eight reconstructed arrays are compared with it.


In [ ]:
INPUT_MODE = "Existing input or snapshot"  # @param ["Existing input or snapshot", "Upload prepared input", "Original files in data/", "Upload original files", "Download original files"]
VCF_URL = "http://solomics.agis.org.cn/tomato/ftp/genotypes/706.sv.vcf.gz"  # @param {type:"string"}
PHENOTYPES_URL = "http://solomics.agis.org.cn/tomato/ftp/expression/pheno_exp_meta.txt"  # @param {type:"string"}

snapshot = Path("weights/_input_snapshot.npz")
output = Path("data/dataset_inputs.npz")
prepared = None
if INPUT_MODE == "Upload prepared input":
    from google.colab import files
    output.parent.mkdir(exist_ok=True)
    if output.exists():
        raise FileExistsError("data/dataset_inputs.npz already exists; choose Existing input or snapshot.")
    with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
        previous = Path.cwd()
        try:
            os.chdir(temporary)
            uploaded = files.upload()
        finally:
            os.chdir(previous)
        if len(uploaded) != 1 or next(iter(uploaded)) != "dataset_inputs.npz":
            raise ValueError("Upload the prepared file named dataset_inputs.npz.")
        with output.open("xb") as handle:
            handle.write(next(iter(uploaded.values())))
    prepared = output
elif INPUT_MODE == "Existing input or snapshot":
    if snapshot.is_file():
        prepared = snapshot
        print("Using the fixed original input supplied with the saved prediction states.")
    elif output.is_file():
        prepared = output
        print("Using the existing prepared input.")
    else:
        raise FileNotFoundError(
            "No prepared input is available. Parameter-only weight ZIPs contain no input data. "
            "Upload dataset_inputs.npz, or select an original-file option for 706.sv.vcf.gz and pheno_exp_meta.txt "
            "using the source links above, then rerun this cell.")
else:
    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)
    names = {"706.sv.vcf.gz", "pheno_exp_meta.txt"}
    if INPUT_MODE == "Upload original files":
        from google.colab import files
        print("Choose 706.sv.vcf.gz and pheno_exp_meta.txt.")
        previous = Path.cwd()
        with tempfile.TemporaryDirectory(dir=REPO_DIR) as temporary:
            try:
                os.chdir(temporary)
                uploaded = files.upload()
            finally:
                os.chdir(previous)
            if not uploaded or not set(uploaded) <= names:
                raise ValueError("Upload only 706.sv.vcf.gz and pheno_exp_meta.txt.")
            for name, payload in uploaded.items():
                target = data_dir / name
                if target.exists():
                    raise FileExistsError(f"{target} already exists; use Original files in data/.")
                with target.open("xb") as handle:
                    handle.write(payload)
    elif INPUT_MODE == "Download original files":
        for name, url in (("706.sv.vcf.gz", VCF_URL), ("pheno_exp_meta.txt", PHENOTYPES_URL)):
            target = data_dir / name
            if not target.exists():
                with tempfile.TemporaryDirectory(dir=data_dir) as temporary:
                    downloaded = Path(temporary) / name
                    download_to(url, downloaded)
                    if target.exists():
                        raise FileExistsError(f"{target} appeared during download.")
                    downloaded.rename(target)
    missing = [name for name in sorted(names) if not (data_dir / name).is_file()]
    if missing:
        raise FileNotFoundError("Place these original files in data/: " + ", ".join(missing))
    if not output.exists():
        subprocess.run([sys.executable, "-m", "src.prepare_data",
                        "--vcf", "data/706.sv.vcf.gz", "--phenotypes", "data/pheno_exp_meta.txt",
                        "--output", str(output)], check=True)
    prepared = output

if prepared is not None:
    expected_keys = {"dosage", "accessions", "traits", "phenotypes", "marker_id", "chrom", "pos", "svlen"}
    with np.load(prepared, allow_pickle=False) as current:
        assert set(current.files) == expected_keys
        np.testing.assert_array_equal(current["accessions"], plan["accessions"])
        np.testing.assert_array_equal(current["traits"], plan["traits"])
        assert current["dosage"].shape == (324, 106362)
        assert current["phenotypes"].shape == (324, 36)
        if snapshot.is_file() and prepared.resolve() != snapshot.resolve():
            with np.load(snapshot, allow_pickle=False) as original:
                assert set(original.files) == expected_keys
                for key in expected_keys:
                    np.testing.assert_array_equal(current[key], original[key], err_msg=key)
            print("All eight reconstructed arrays match the independent original snapshot.")
    print("Benchmark input ready:", prepared)


## Train (optional)

For training, select **Runtime → Change runtime type → GPU**. Enable `RUN_TRAINING` and enter a trait ID; `TRAIN_TRAIT="all"` fits all 36 traits using the fixed protocol. New outputs include complete `.full.pt` checkpoints with AdamW state, prediction states and `.json` records. Resume verifies and skips completed cases.


In [ ]:
RUN_TRAINING = False  # @param {type:"boolean"}
TRAIN_TRAIT = "SlFM0030"  # @param {type:"string"}

if RUN_TRAINING:
    if prepared is None:
        raise RuntimeError("Choose or prepare the input arrays before training.")
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    subprocess.run([sys.executable, "-m", "src.train", "--model", MODEL,
                    "--trait", TRAIN_TRAIT, "--fold", "0", "--device", device,
                    "--data", str(prepared), "--output", "results", "--save-optimizer", "--resume"], check=True)
else:
    print("Training disabled. Enable RUN_TRAINING to fit the selected traits.")


## Predict

Automatic selection uses a newly trained checkpoint first, then the selected pretrained weights. For parameter-only weights, the fixed reference input restores the training preprocessing and GP state without retraining the neural network. Prediction exports all supplied accessions to a CSV file.


In [ ]:
PREDICTION_SOURCE = "Automatic"  # @param ["Automatic", "Training output", "Pretrained weights"]
PREDICT_TRAIT = "SlFM0030"  # @param {type:"string"}


def find_state(root, trait):
    if trait not in plan["traits"]:
        raise ValueError("Trait is not in the fixed 36-trait experiment.")
    root = Path(root)
    candidates = [root / MODEL / f"{trait}.pt",
                  root / MODEL / "fold_0" / f"{MODEL}_f0_{trait}.full.pt",
                  root / "rebuilt_fold0" / MODEL / "fold_0" / f"{MODEL}_f0_{trait}.full.pt"]
    return next((path for path in candidates if path.is_file()), None)


sources = ["results", "weights"] if PREDICTION_SOURCE == "Automatic" else [
    "results" if PREDICTION_SOURCE == "Training output" else "weights"]
checkpoint = None
for source in sources:
    checkpoint = find_state(source, PREDICT_TRAIT)
    if checkpoint is not None:
        break
if prepared is None:
    raise RuntimeError("Complete Input data before prediction.")
if checkpoint is None:
    raise FileNotFoundError(f"No {MODEL} weights for {PREDICT_TRAIT}. Upload {MODEL}_weights.zip or complete training for this trait.")
fit_label = "pretrained" if source == "weights" else "training_output"
basename = f"{fit_label}_{MODEL}_{PREDICT_TRAIT}"
output = Path("predictions") / (basename + ".csv")
run_number = 2
while output.exists():
    output = Path("predictions") / f"{basename}_run{run_number}.csv"
    run_number += 1
subprocess.run([sys.executable, "-m", "src.predict", "--checkpoint", str(checkpoint),
                "--data", str(prepared), "--model", MODEL, "--trait", PREDICT_TRAIT,
                "--device", "cpu", "--output", str(output)], check=True)
print("Prediction file:", output)
import csv
with output.open(newline="") as handle:
    for row in list(csv.reader(handle))[:6]:
        print(row)


## Evaluate (optional)

Enable evaluation after loading or training all 36 traits for the selected capacity. The scorer uses the fixed held-out accessions and reports per-trait metrics and their equal-weight mean.


In [ ]:
RUN_SCORING = False  # @param {type:"boolean"}
SCORE_SOURCE = "Pretrained weights"  # @param ["Pretrained weights", "Training output"]

if RUN_SCORING:
    if prepared is None:
        raise RuntimeError("Complete Input data before evaluation.")
    result_root = Path("results") if SCORE_SOURCE == "Training output" else Path("weights")
    if result_root.name == "weights" and not (result_root / MODEL).is_dir():
        result_root = result_root / "rebuilt_fold0"
    missing = [trait for trait in plan["traits"] if find_state(result_root, trait) is None]
    if missing:
        raise FileNotFoundError(f"Missing {len(missing)} trait weights: {', '.join(missing)}. Load {MODEL}_weights.zip or complete training.")
    score_label = "training_output" if SCORE_SOURCE == "Training output" else "pretrained"
    output = Path(f"metrics_{score_label}_{MODEL}.json")
    if output.exists():
        raise FileExistsError(f"{output} already exists. Save or remove it before running evaluation again.")
    subprocess.run([sys.executable, "-m", "src.score", "--results", str(result_root),
                    "--model", MODEL, "--data", str(prepared), "--output", str(output)], check=True)
else:
    print("Evaluation disabled. Enable RUN_SCORING to evaluate all 36 traits.")


Download the prediction CSV and any new training outputs before disconnecting the runtime.
